# 09_04 On your own: a better translator

**The brief.** Kittiwake Mobile wants its short in-app messages drafted in Spanish by a model it can run
and retrain on its own small server. The lab's translator is the starting point. Make it better:

- start from `translate.lab_model()`, which has had 1,392 training steps (two passes over the pairs, in
  batches of 128);
- a budget of **700 more training steps** (`translate.BUDGET_STEPS`), about one more pass: pass
  `max_steps=700` or less to `translate.train`, which stops there and says so. 700 steps take a few
  minutes in a session. The model counts its own steps, `save_model` records them, and the checkpoint
  refuses a model with more than 1,392 + 700 in all, however you got there;
- decode with a **beam of at most 5** (`beam_size` from 1 to 5);
- save with `translate.save_model(model, "out/translator.pt", beam_size=k, **kwargs)`, where `kwargs` are
  the arguments that rebuild the model (for the lab's model, none);
- save a short report, `out/translator_report.json`: what you changed, the learning rate, the steps
  beyond the lab model's, the beam size and your own chrF.

**The bar.** chrF at least **46.0** on the 1,174 held-out sentences of `translate.eval_split`, decoded with
your saved beam size. The checkpoint rebuilds your model and scores it itself.

**Rules of the road.** Any class in `translate.py` is fine, and so is one you add to that file (a class
defined in this notebook cannot be rebuilt by the checkpoint). Fix a seed. The lab's model was trained at a
constant learning rate of 0.002 and never at anything lower.

There is no worked example here. `check_on_your_own()` scores your saved model exactly as the checkpoint will.

Running this in Google Colab? This cell sets it up; in CourseLabs it does nothing.

In [ ]:
# Colab setup. In a CourseLabs session this cell does nothing.
import os, sys
if "google.colab" in sys.modules:
    import importlib, importlib.util, subprocess
    LAB, REPO = "lab-nlp-09-from-one-language-to-another", "/content/nlp-course"
    if not os.path.isdir(REPO):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/fenago/nlp-course.git", REPO], check=True)
    os.chdir(f"{REPO}/{LAB}")
    if not os.path.exists("data"):
        os.symlink("../data", "data")
    os.makedirs("out", exist_ok=True)
    os.environ["NLPLAB_DATA"] = f"{REPO}/data"
    sys.path.insert(0, os.getcwd())
    PIP = {'torch': 'torch'}
    missing = [spec for mod, spec in PIP.items() if importlib.util.find_spec(mod) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
        importlib.invalidate_caches()
    print(f"Ready: {LAB} and its data are in {os.getcwd()}; installed {len(missing)} package(s).")
elif not os.path.isdir("/opt/nlplab/data") and os.path.isdir("data"):
    # A downloaded copy on your own computer: the helpers read data/ from here.
    os.environ["NLPLAB_DATA"] = os.path.abspath("data")

In [ ]:
import json
import time
import torch
import translate
from nlpcheck import check_on_your_own

torch.set_num_threads(4)
data = translate.load_split()
src, refs = translate.eval_split(data)
print(len(data["train"]), "training pairs,", len(src), "held-out sentences to score on")

In [ ]:
torch.manual_seed(0)
t = time.time()
model = None     # YOUR CODE HERE: start from translate.lab_model() and improve it within the budget
train_seconds = time.time() - t
print(f"trained in {train_seconds:.0f} s")

In [ ]:
beam_size = 1    # YOUR CODE HERE: choose 1 to 5
t = time.time()
score = translate.chrf(translate.translate_all(model, src, data, k=beam_size), refs) if model is not None else None
print("chrF", score, f"({time.time() - t:.0f} s to translate)")
# YOUR CODE HERE: translate.save_model(model, "out/translator.pt", beam_size=beam_size)
steps = getattr(model, "steps_trained", 0) - translate.LAB_STEPS
print("training steps beyond the lab model's:", steps, "of", translate.BUDGET_STEPS)
json.dump({"changes": None, "lr": None, "train_steps": steps, "train_seconds": round(train_seconds, 1),
           "beam_size": beam_size, "chrf": score}, open("out/translator_report.json", "w"), indent=1)

In [ ]:
check_on_your_own()

When the check prints `Right:`, go back to the lab instructions for the sign-off and the checkpoint.